In [ ]:
from pathlib import Path
import json

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from transformers import PreTrainedTokenizerFast

In [ ]:
MODEL_DIR = Path("./model")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILE = "./data.txt"

VOCAB_SIZE = 32000
MIN_FREQUENCY = 2

SPECIAL_TOKENS = [
    '<|unk|>',
    '<|pad|>',
    "<|bos|>",
    "<|eos|>",
]

In [ ]:
base_tokenizer = Tokenizer(BPE(unk_token='<|unk|>'))
base_tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=True)
base_tokenizer.decoder = ByteLevelDecoder()

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    initial_alphabet=ByteLevel().alphabet(),
    special_tokens=SPECIAL_TOKENS
)

base_tokenizer.train(files=[DATA_FILE], trainer=trainer)

# Save raw model once
base_tokenizer.model.save(str(MODEL_DIR))
base_tokenizer.save(str(MODEL_DIR / "tokenizer.json"))


In [ ]:
fast_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=str(MODEL_DIR / "tokenizer.json"),
    unk_token='<|unk|>',
    bos_token="<|bos|>",
    eos_token="<|eos|>",
    pad_token="<|pad|>",
    model_input_names=["input_ids", 'attention_mask']
)

fast_tokenizer.save_pretrained(str(MODEL_DIR))


In [ ]:
test_text =  '<?php $str = "PHP,MySQL,HTML"; print_r(explode(",",$str)); ?> | Prints Array ( [0] => PHP [1] => MySQL [2] => HTML )'


print("Decoded:", fast_tokenizer.decode(fast_tokenizer.encode(test_text)))
print("LEN:", len(fast_tokenizer.encode(test_text)))
print("Base vocab size:", fast_tokenizer.vocab_size)
print("Total length (with special):", len(fast_tokenizer))